# ch05 Bonus 18：LLM Zoo 总览对比

> 对照官方 `ch05` 各 LLM 子目录汇总

## 一句话

把前面 10-17 学的所有模型（Llama/Qwen3/Gemma3/Gemma4/OLMo3/TinyAya）放在一张表里，对比架构差异，建立「现代开源 LLM 设计选择」的全景认知。

## 现代 LLM 的共同基础（Llama 范式）

几乎所有 2023+ 的开源 LLM 都基于：**Decoder-only Transformer + RoPE + RMSNorm + SwiGLU + GQA**。差异在于各自的微调改进。

## 架构对比总表

| 模型 | 位置编码 | 归一化 | FFN | 注意力变体 | 特色改进 |
|------|---------|--------|-----|-----------|---------|
| **GPT-2**（基线） | 绝对嵌入 | LayerNorm | GELU | MHA | — |
| **Llama-3** | RoPE | RMSNorm | SwiGLU | GQA | 现代基线 |
| **Qwen3** | RoPE | RMSNorm | SwiGLU | GQA + **QK-Norm** | 训练稳定性 |
| **Gemma3** | RoPE | RMSNorm | SwiGLU(GELU) | GQA + **SWA**(5:1) + QK-Norm | 长文本+缩放emb |
| **OLMo3** | RoPE | RMSNorm | SwiGLU | GQA + QK-Norm | **完全开源**+可学习缩放 |
| **TinyAya** | RoPE | RMSNorm | SwiGLU | MHA/GQA | **多语言分词** |

> **共性**：RoPE + RMSNorm + SwiGLU + GQA 已是事实标准。
> **个性**：2024-2025 的改进集中在 QK-Norm（稳定性）、SWA（长文本）、muP（超参迁移）。

## 关键改造点的演进逻辑

In [ ]:
# 用代码直观对比各模型的「注意力分数缩放」策略差异
import torch

head_dim = 128
scores = torch.randn(1, 8, 16, 16)  # 模拟注意力分数

print("各模型对注意力分数的处理方式：")
print("-" * 60)

strategies = {
    "GPT-2 / Llama": scores / (head_dim ** 0.5),
    "OLMo3 (可学习)": scores * torch.tensor(head_dim ** -0.5),  # scale 可训练
    "Qwen3/Gemma3 (QK-Norm 先归一化)": scores / (head_dim ** 0.5),  # q,k 已先 RMSNorm
}
for name, s in strategies.items():
    print(f"  {name:<35} → 分数方差 {s.var():.4f}")

print("\n💡 表面看输出方差接近，但 QK-Norm 是在 q/k 进入分数前就约束了范围，")
print("   从根本上防止「某些头范数爆炸→注意力过尖」的问题，这是 2024 年的关键发现。")

## 学习路径总结

1. **ch03-04**：先掌握标准注意力 + GPT
2. **ch04 bonus**：学注意力变体（GQA/MLA/SWA/MoE 等）
3. **ch05 主线**：学会训练（loss、优化器、加载权重）
4. **ch05 bonus 10-17**：把这些组件拼成真实 LLM，理解工业界的设计选择

之后进入 ch06（分类微调）和 ch07（指令微调），让模型从「只会续写」变成「会回答、会遵循指令」。

---
> 📌 这是 ch05 bonus 的总结篇。各模型的完整实现见对应的前序 notebook（10-17）。